**PP434 portfolio** · Jakob Kilian Bilan · CC2, chart 1

# WID top 1% income share: download

Source: World Inequality Database via the Our World in Data chart API (`.csv` endpoint). Output: `cc2a_wid_top1_pretax.csv` (columns `country, code, year, share`).

In [ ]:
import io
import requests
import pandas as pd

URL = "https://ourworldindata.org/grapher/income-share-top-1-before-tax-wid.csv?v=1&csvType=full&useColumnShortNames=false"
HEADERS = {"User-Agent": "Our World In Data data fetch/1.0"}   # header used in OWID's own code examples
COUNTRIES = {"USA": "United States", "GBR": "United Kingdom", "FRA": "France"}
OUTPUTS = ["cc2a_wid_top1_pretax.csv"]

In [ ]:
response = requests.get(URL, headers=HEADERS, timeout=60)
print("HTTP", response.status_code)
response.raise_for_status()
raw = pd.read_csv(io.StringIO(response.text))
print(raw.shape)
print(raw.columns.tolist())   # expected: Entity, Code, Year, then the indicator
raw.head()

In [ ]:
# The indicator column is the first numeric column after Entity, Code, Year (its name can change between releases)
value_col = [c for c in raw.columns if c not in ("Entity", "Code", "Year") and pd.api.types.is_numeric_dtype(raw[c])][0]
print("indicator column:", value_col)

df = raw[raw["Code"].isin(COUNTRIES)].rename(columns={"Code": "code", "Year": "year", value_col: "share"})
df["country"] = df["code"].map(COUNTRIES)
if df["share"].max() <= 1:          # shares stored as fractions -> percent
    df["share"] = df["share"] * 100
df = df[["country", "code", "year", "share"]].dropna().sort_values(["country", "year"]).reset_index(drop=True)

# Checks
assert set(df["country"]) == set(COUNTRIES.values()), "a country is missing"
assert df["share"].between(0, 60).all(), "shares outside 0-60%: check the units"
print(df.groupby("country")["year"].agg(["min", "max", "count"]))
print(df.groupby("country").tail(1))

In [ ]:
df.to_csv(OUTPUTS[0], index=False)
print("saved", OUTPUTS[0], len(df), "rows")

In [ ]:
# Colab only: download the output to your computer, then upload it to the data folder of your GitHub repository
try:
    from google.colab import files
    for f in OUTPUTS:
        files.download(f)
except ImportError:
    print("Not running in Colab: the files are in the working folder:", OUTPUTS)